# 03 · Chạy Qwen 7B trên bộ pilot và bộ câu TV1 (Tuần 3 · TV2)

**Chạy ở đâu:** Google Colab, **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**.

**Làm gì:** phục vụ `Qwen/Qwen2.5-7B-Instruct-AWQ` bằng vLLM rồi chạy `rag.run_inference`, temperature 0:
- Phần A (ưu tiên): bộ pilot 80 câu, C0 (không truy xuất), C2 (RAG vector), C3 (kết hợp + reranker), C4 (C3 + prompt mốc thời gian).
- Phần B: 168 câu mới của TV1 (`tv1_v0.3.jsonl`), C0, C2, C3, C4.
- Phần C: lần chạy 2 và 3 trên bộ pilot (phiếu phân công: 3 lần mỗi câu).

Ngữ cảnh RAG đã truy xuất sẵn trên máy TV2 (`outputs/contexts/`), Colab chỉ sinh câu trả lời. Câu đã chạy (ví dụ C0, C2 pilot hôm 26/11) được bỏ qua nhờ file đầu ra trong repo.

**Kết quả:** ô cuối tải về `qwen_week3_outputs.zip`. Hết giờ GPU giữa chừng: chạy ô cuối để tải phần đã xong, lần sau mở lại notebook và chạy tiếp (các câu đã có sẽ được bỏ qua sau khi TV2 đưa file lên repo).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
%%capture
!pip install vllm
!git clone -b an-huy https://github.com/yuhna111/NEXUS2026-FeelTheBeat repo
!pip install -e repo

In [ ]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # trùng snapshot của qwen-7b trong configs/models.yaml
log = open("vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", "8000",
     "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"],
    stdout=log, stderr=subprocess.STDOUT,
)

t0 = time.time()
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM dừng sớm — xem vllm.log:\n" + open("vllm.log").read()[-3000:])
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
print(f"Máy chủ sẵn sàng sau {time.time() - t0:.0f} giây")

In [ ]:
import os
os.chdir("repo")
os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["LLM_CACHE_DIR"] = ".llm_cache"
P = "data/benchmark/pilot_80_v0.3.jsonl"
T = "data/benchmark/tv1_v0.3.jsonl"
!ls outputs/contexts/*/

In [ ]:
# Phần A: pilot 80 câu, C0 / C2 / C3 / C4 (~5–10 phút mỗi cấu hình chưa chạy)
for c in ["C0", "C2", "C3", "C4"]:
    !python -m rag.run_inference --config {c} --bench $P --models qwen-7b

In [ ]:
# Phần B: 168 câu của TV1, C0 / C2 / C3 / C4 (~10–15 phút mỗi cấu hình)
# Lấy ngữ cảnh mới nhất TV2 vừa đưa lên (C2/C3 cho bộ TV1); thiếu thì cấu hình đó báo lỗi, các cấu hình khác vẫn chạy
!git pull -q origin an-huy
for c in ["C0", "C2", "C3", "C4"]:
    !python -m rag.run_inference --config {c} --bench $T --models qwen-7b

In [ ]:
# Phần B2: 60 câu mới của TV3 (tv3_v0.3), C0 / C2 / C3 / C4 (~5 phút mỗi cấu hình)
!git pull -q origin an-huy
V = "data/benchmark/tv3_v0.3.jsonl"
for c in ["C0", "C2", "C3", "C4"]:
    !python -m rag.run_inference --config {c} --bench $V --models qwen-7b

In [ ]:
# Phần C: lần 2 và 3 trên pilot (seed = số lần chạy)
for c in ["C0", "C2", "C3", "C4"]:
    !python -m rag.run_inference --config {c} --bench $P --models qwen-7b --runs 3 --first-run 2

In [ ]:
# Thống kê nhanh: số câu và token ra mỗi file
import json, glob
for f in sorted(glob.glob("outputs/*/*/qwen-7b*.jsonl")):
    rows = [json.loads(l) for l in open(f, encoding="utf-8")]
    toks = [r["usage"].get("completion_tokens", 0) for r in rows]
    print(f, len(rows), "câu ·", sum(toks), "token ra")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
server.terminate()
!zip -qr /content/qwen_week3_outputs.zip outputs/pilot_80_v0.3 outputs/tv1_v0.3 outputs/tv3_v0.3 .llm_cache
from google.colab import files
files.download("/content/qwen_week3_outputs.zip")